# 00 · Cek Environment Ryu + Mininet

Notebook ini memeriksa apakah environment hasil `setup_env.sh` / `setup_env.bat` sudah siap, lalu menjalankan **smoke test**: controller Ryu `simple_switch_13` + topologi Mininet kecil + `pingAll`.

Semua notebook pada proyek ini punya dua mode:

| Mode | Kapan aktif | Isi |
|---|---|---|
| `mininet` | Linux/WSL2 + kernel berjalan sebagai **root** (`bash run_jupyter.sh`) + Mininet & Ryu terpasang | Emulasi nyata: Open vSwitch, link `tc` (delay/bandwidth/loss), controller Ryu, trafik ping/iperf/hping3 |
| `sim` | selain itu (mis. Windows, atau kernel bukan root) | Model simulasi murni Python dengan skema data yang **sama**, sehingga alur analisis tetap bisa dicoba |

In [1]:
import os, sys, platform, shutil, subprocess, importlib, time
from pathlib import Path

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNTIME = BASE / "runtime" / "00_cek"
RUNTIME.mkdir(parents=True, exist_ok=True)
OF_PORT = 6653

def ryu_manager_path():
    cand = Path(sys.executable).parent / "ryu-manager"
    return str(cand) if cand.exists() else shutil.which("ryu-manager")

print("Folder proyek :", BASE)
print("Python        :", sys.version.split()[0], "→", sys.executable)
print("Platform      :", platform.platform())
print("User root     :", hasattr(os, "geteuid") and os.geteuid() == 0)

Folder proyek : /mnt/c/Users/wirar/OneDrive/Documents/GitHub/sdn
Python        : 3.9.25 → /home/wirar/.venvs/sdn-ryu-mininet/bin/python
Platform      : Linux-6.18.58.1-microsoft-standard-WSL2-x86_64-with-glibc2.43
User root     : True


## 1. Versi library Python

In [2]:
rows = []
for mod in ["ryu", "eventlet", "mininet.net", "networkx", "numpy", "pandas", "sklearn", "matplotlib"]:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", None)
        if mod == "mininet.net":
            ver = getattr(m, "VERSION", "ok")
        rows.append((mod, "✔", ver or "ok"))
    except Exception as exc:
        rows.append((mod, "✘", f"{type(exc).__name__}: {exc}"[:80]))
for r in rows:
    print(f"{r[1]} {r[0]:<12} {r[2]}")

if platform.system() == "Windows":
    print("\nℹ Kernel ini berjalan di WINDOWS: Ryu & Mininet memang tidak dipasang di sini (Mininet hanya jalan di Linux).")
    print("  Untuk emulasi asli jalankan 'setup_env.bat wsl' lalu 'run_jupyter.bat wsl'. Di Windows notebook memakai MODE='sim'.")

if sys.version_info[:2] >= (3, 10) and any(r[0] == "ryu" and r[1] == "✔" for r in rows):
    print("\n⚠ Ryu terimpor di Python >= 3.10 — umumnya ryu-manager gagal (eventlet). Gunakan venv Python 3.9 dari setup_env.sh.")

✔ ryu          ok
✔ eventlet     0.30.2
✔ mininet.net  2.3.0
✔ networkx     3.2.1
✔ numpy        2.0.2
✔ pandas       2.3.3
✔ sklearn      1.6.1
✔ matplotlib   3.9.4


## 2. Tool sistem (Mininet, Open vSwitch, trafik generator)

In [3]:
def run(cmd):
    try:
        out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=15)
        return (out.stdout or out.stderr).strip().splitlines()[0] if (out.stdout or out.stderr) else ""
    except Exception as exc:
        return f"error: {exc}"

checks = {
    "mn":          "mn --version",
    "ovs-vsctl":   "ovs-vsctl --version",
    "mnexec":      "which mnexec",
    "iperf":       "iperf -v 2>&1",
    "hping3":      "which hping3",
    "curl":        "curl --version",
    "ryu-manager": f"{ryu_manager_path()} --version 2>&1" if ryu_manager_path() else "false",
}
for name, cmd in checks.items():
    found = shutil.which(name) or (name == "ryu-manager" and ryu_manager_path())
    print(f"{'✔' if found else '✘'} {name:<12} {run(cmd) if found else 'tidak ditemukan'}")

if platform.system() == "Linux":
    ovs_kernel = os.path.isdir("/sys/module/openvswitch")
    if not ovs_kernel and os.geteuid() == 0 and shutil.which("modprobe"):
        ovs_kernel = subprocess.run(["modprobe", "openvswitch"], capture_output=True).returncode == 0
    OVS_DATAPATH = "kernel" if ovs_kernel else "user"
    print(f"\nDatapath OVS yang akan dipakai: {OVS_DATAPATH}")
    if os.geteuid() == 0:
        print("Status OVS:", run("ovs-vsctl show") or "OK (belum ada bridge)")
else:
    OVS_DATAPATH = "user"

✔ mn           2.3.0
✔ ovs-vsctl    ovs-vsctl (Open vSwitch) 3.7.1
✔ mnexec       /usr/bin/mnexec
✔ iperf        iperf version 2.2.1 (4 Nov 2024) pthreads
✔ hping3       /usr/sbin/hping3
✔ curl         curl 8.18.0 (x86_64-pc-linux-gnu) libcurl/8.18.0 OpenSSL/3.5.5 zlib/1.3.1 brotli/1.2.0 zstd/1.5.7 libidn2/2.3.8 libpsl/0.21.2 libssh2/1.11.1 nghttp2/1.68.0 librtmp/2.3 mit-krb5/1.22.1 OpenLDAP/2.6.10
✔ ryu-manager  ryu-manager 4.34

Datapath OVS yang akan dipakai: kernel
Status OVS: 94964751-4d04-48ee-9768-9e9965ffb44c


## 3. Penentuan mode

In [4]:
def detect_mode():
    if platform.system() != "Linux":
        return "sim", "OS bukan Linux (Mininet hanya jalan di Linux/WSL2)"
    try:
        import mininet.net  # noqa: F401
    except Exception:
        return "sim", "library mininet tidak ditemukan di venv"
    if os.geteuid() != 0:
        return "sim", "kernel tidak berjalan sebagai root — jalankan Jupyter via: bash run_jupyter.sh"
    if not shutil.which("ovs-vsctl"):
        return "sim", "Open vSwitch tidak ditemukan"
    if not ryu_manager_path():
        return "sim", "ryu-manager tidak ditemukan"
    return "mininet", "Linux + root + Mininet + Ryu tersedia"

MODE, reason = detect_mode()
print(f"MODE = {MODE!r}  ({reason})")

MODE = 'mininet'  (Linux + root + Mininet + Ryu tersedia)


## 4. Smoke test: Ryu `simple_switch_13` + Mininet `pingAll`
Topologi: 1 switch OpenFlow 1.3, 3 host. Controller Ryu dijalankan sebagai proses terpisah (`ryu-manager`) yang mendengarkan port 6653, lalu Mininet menghubungkan switch ke `RemoteController`.

In [5]:
if MODE != "mininet":
    print("Smoke test dilewati (MODE='sim'). Notebook 01 & 02 tetap bisa dijalankan dalam mode simulasi.")
else:
    from functools import partial
    from mininet.net import Mininet
    from mininet.node import RemoteController, OVSSwitch
    from mininet.topo import SingleSwitchTopo
    from mininet.clean import cleanup
    from mininet.log import setLogLevel

    setLogLevel("warning")
    cleanup()
    subprocess.run(["fuser", "-k", f"{OF_PORT}/tcp"], capture_output=True)
    log = open(RUNTIME / "ryu_smoke.log", "w")
    ryu = subprocess.Popen([ryu_manager_path(), "--ofp-tcp-listen-port", str(OF_PORT), "ryu.app.simple_switch_13"],
                           stdout=log, stderr=subprocess.STDOUT)
    time.sleep(3)
    try:
        net = Mininet(topo=SingleSwitchTopo(3),
                      switch=partial(OVSSwitch, protocols="OpenFlow13", datapath=OVS_DATAPATH),
                      controller=partial(RemoteController, ip="127.0.0.1", port=OF_PORT))
        net.start()
        time.sleep(2)
        loss = net.pingAll(timeout=2)
        print(f"\npingAll packet loss: {loss:.0f}%  →", "LULUS ✔" if loss == 0 else "GAGAL ✘ (lihat runtime/00_cek/ryu_smoke.log)")
        print(net.get("s1").cmd("ovs-ofctl -O OpenFlow13 dump-flows s1"))
    finally:
        try:
            net.stop()
        except Exception:
            pass
        ryu.terminate(); ryu.wait(timeout=5)
        cleanup()


pingAll packet loss: 0%  → LULUS ✔
 cookie=0x0, duration=0.021s, table=0, n_packets=2, n_bytes=196, priority=1,in_port="s1-eth2",dl_src=be:8b:f0:0e:0b:e7,dl_dst=1a:b8:48:ea:91:87 actions=output:"s1-eth1"
 cookie=0x0, duration=0.020s, table=0, n_packets=2, n_bytes=196, priority=1,in_port="s1-eth1",dl_src=1a:b8:48:ea:91:87,dl_dst=be:8b:f0:0e:0b:e7 actions=output:"s1-eth2"
 cookie=0x0, duration=0.016s, table=0, n_packets=2, n_bytes=196, priority=1,in_port="s1-eth3",dl_src=ae:f8:e6:d1:33:88,dl_dst=1a:b8:48:ea:91:87 actions=output:"s1-eth1"
 cookie=0x0, duration=0.015s, table=0, n_packets=1, n_bytes=98, priority=1,in_port="s1-eth1",dl_src=1a:b8:48:ea:91:87,dl_dst=ae:f8:e6:d1:33:88 actions=output:"s1-eth3"
 cookie=0x0, duration=0.009s, table=0, n_packets=1, n_bytes=98, priority=1,in_port="s1-eth3",dl_src=ae:f8:e6:d1:33:88,dl_dst=be:8b:f0:0e:0b:e7 actions=output:"s1-eth2"
 cookie=0x0, duration=0.008s, table=0, n_packets=0, n_bytes=0, priority=1,in_port="s1-eth2",dl_src=be:8b:f0:0e:0b:e7,dl_d